# Week 23: Next-Gen AI Reasoning, MCTS & Test-Time Compute
## Notebook 03: Self-Consistency & Majority Voting from Scratch

### 1. Conceptual Intuition & Real-World Use Case
When prompting an LLM with Chain-of-Thought (CoT) at temperature $T=0$ (greedy decoding):
- The model follows a single deterministic reasoning path.
- If it makes one slight error early on, the answer is wrong.

In 2022, Wang et al. (Google Brain) introduced **Self-Consistency**:
- Complex reasoning problems admit multiple distinct valid paths to the correct answer.
- Sample $N$ independent reasoning paths at temperature $T > 0$ (e.g. $T=0.7$).
- Extract the final answers and take the **Majority Vote**!
- Boosts accuracy by **10% to 20% across GSM8K and SVAMP reasoning benchmarks** without any model fine-tuning!

---

### 2. Mathematical Foundations: Marginalization over Reasoning Paths
Let $Q$ be the question, $r_i$ be sampled reasoning paths, and $a_i$ be the final answer:

$$a^* = \arg\max_a \sum_{i=1}^N \mathbb{I}(a_i = a)$$

#### Normalized Entropy Confidence Score:
Let $p(a)$ be the empirical fraction of paths supporting answer $a$:
$$H = -\sum_a p(a) \log_2 p(a)$$
- Low Entropy ($H \approx 0$): High model consensus / confidence.
- High Entropy ($H > 1.5$): High uncertainty / conflicting reasoning paths.

In [ ]:
from collections import Counter
import numpy as np

# Step 1: Simulate 10 Sampled Chain-of-Thought Responses for a Logic Puzzle
# Simulated paths with slight wording and arithmetic diversity
sampled_responses = [
    {"path_id": 1, "reasoning": "4 shirts * $15 = $60, minus $10 coupon = $50", "answer": 50},
    {"path_id": 2, "reasoning": "Each shirt is 15. Total 60. Take away 10 discount. Result 50.", "answer": 50},
    {"path_id": 3, "reasoning": "4 * 15 = 60. Then subtract 10 gives 50.", "answer": 50},
    {"path_id": 4, "reasoning": "4 * 15 = 60. Plus 10 tax gives 70.", "answer": 70}, # Misunderstood coupon as tax!
    {"path_id": 5, "reasoning": "Total price is 4 x 15 = 60. Discount 10. Leaves 50.", "answer": 50},
    {"path_id": 6, "reasoning": "15 * 4 = 60. Minus 10 = 50 dollars.", "answer": 50},
    {"path_id": 7, "reasoning": "4 shirts at 15 is 60. Discount 10 -> 50.", "answer": 50},
    {"path_id": 8, "reasoning": "4 * 15 = 60. Minus 10 gives 50.", "answer": 50},
    {"path_id": 9, "reasoning": "4 * 15 is 60. With 10 percent discount = 54.", "answer": 54}, # Misunderstood $10 as 10%
    {"path_id": 10, "reasoning": "4 shirts: 60 total. 10 off = 50.", "answer": 50}
]

# Step 2: Implement Majority Voting & Consensus Statistics
answers = [r["answer"] for r in sampled_responses]
counts = Counter(answers)
total_samples = len(answers)

majority_answer, majority_count = counts.most_common(1)[0]
confidence = majority_count / total_samples

# Compute Entropy
entropy = -sum((cnt / total_samples) * np.log2(cnt / total_samples) for cnt in counts.values())

print("--- Self-Consistency Voting Aggregator ---")
print("Sampled Answer Distribution:")
for ans, count in counts.items():
    print(f"  Answer {ans}: {count}/{total_samples} votes ({count/total_samples * 100:.1f}%)")

print(f"\nMajority Voted Consensus Answer: {majority_answer}")
print(f"Consensus Confidence:            {confidence * 100:.1f}%")
print(f"Reasoning Entropy (Uncertainty): {entropy:.3f} bits (Low = Strong agreement)")

### 3. Step-by-Step Code Walkthrough
1. **Diversity Sampling:** Sampling at non-zero temperature produces different linguistic and mathematical derivations.
2. **Noise Cancellation:** Paths 4 and 9 contained reasoning errors, but were overwhelmed by the 8 paths that correctly converged to 50.
3. **Calibrated Confidence:** The percentage of votes ($80\%$) provides a natural uncertainty estimate without external models.

---

### 4. Industry Applications & Trade-offs
- **Medical & Legal AI:** Running 10-20 self-consistency paths on critical queries to eliminate hallucinations.
- **Trade-off:** Multiplies inference cost and latency by $N$ (generating 10 paths costs 10x more tokens).

---

### 5. Top Interview Questions & Answers
**Q1: Why does Self-Consistency improve reasoning performance over standard greedy decoding?**
> **Answer:** In greedy decoding ($T=0$), the model commits to the single most probable first token, which can lead down an erroneous branch. Sampling with $T>0$ explores multiple paths; because correct logical reasoning typically converges to the same invariant answer while errors scatter randomly, majority voting acts as an effective noise filter.

**Q2: What is Weighted Self-Consistency?**
> **Answer:** Rather than counting each vote as 1, Weighted Self-Consistency weights each path by its perplexity or by a Process Reward Model score: $a^* = \arg\max_a \sum_{i \in \text{paths}(a)} \text{score}(r_i)$, giving higher influence to logically sound solutions.